# **AquaShack Custom Transforms**

## **Purpose**

#### Implementations of the custom transforms referenced from `data/meta-data/meta_data.json` via the `custom_transform` property. The registry primitives (`CUSTOM_TRANSFORMS`, `register_custom_transform`, `run_custom_transform`) live in `AquaShack_functions`; this notebook only holds the decorated transform functions.

#### The Bronze to Silver notebook runs `%run AquaShack_functions` first (so the decorator exists), then `%run AquaShack_custom_transforms` (so each `@register_custom_transform(...)` below populates the registry before the orchestration loop dispatches).

#### A custom transform is a pure `(df, meta_entity) -> df` function. The caller still owns the write, so all write policy stays in `write_to_delta_overwrite`.


## **Imports**


In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


## **Transform: enrich_transactions**

#### Default Bronze to Silver would just dedupe and fill nulls. Transactions also needs a derived `TransactionDateId` (yyyyMMdd int) for joins against `dim_dates` downstream, plus a `LoadedAt` audit column.


In [ ]:
@register_custom_transform("enrich_transactions")
def enrich_transactions(df: DataFrame, meta_entity: dict) -> DataFrame:
    df = df.dropDuplicates()
    df = df.na.fill(0)
    df = df.na.fill("N/A")

    if "TransactionDate" in [c[0] for c in df.dtypes]:
        df = df.withColumn(
            "TransactionDateId",
            F.expr("cast(date_format(to_date(TransactionDate), 'yyyyMMdd') as int)")
        )

    df = df.withColumn("LoadedAt", F.current_timestamp())
    return df


## **Transform: count_cows_from_overview**

#### `Cow_Camera_Overviews` is not tabular: PNGs under `year=YYYY/month=MM/day=DD`. The image reader loads them as binary; this transform calls Azure AI Vision `AnalyzeImage` via SynapseML and reduces each image to a row with `id`, `cow_count`, and the partition columns.


In [ ]:
@register_custom_transform("count_cows_from_overview")
def count_cows_from_overview(df: DataFrame, meta_entity: dict) -> DataFrame:
    """
    Count cows in field overview images using Azure AI Vision (AnalyzeImage
    REST API) called directly from the driver. No Spark UDF.

    Emits one row per image: id, cow_count, year, month, day.
    """
    import requests
    from pyspark.sql.types import StructType, StructField, IntegerType

    service_key = "<insert-from-azure>"
    end_point = "<insert-endpoint-from-azure>"
    analyze_url = f"{end_point.rstrip('/')}/vision/v3.2/analyze?visualFeatures=Objects"

    cow_like = {"cow", "mammal", "animal"}

    rows = df.select("url", "content", "year", "month", "day").collect()

    results = []
    for r in rows:
        filename = r["url"].split("/")[-1]
        id_value = int(filename.split("_")[1][:4])

        response = requests.post(
            analyze_url,
            headers={
                "Ocp-Apim-Subscription-Key": service_key,
                "Content-Type": "application/octet-stream",
            },
            data=bytes(r["content"]),
            timeout=30,
        )
        response.raise_for_status()
        objects = response.json().get("objects", [])
        cow_count = sum(1 for o in objects if o.get("object", "").lower() in cow_like)

        results.append((id_value, cow_count, r["year"], r["month"], r["day"]))

    schema = StructType([
        StructField("id", IntegerType(), False),
        StructField("cow_count", IntegerType(), False),
        StructField("year", IntegerType(), True),
        StructField("month", IntegerType(), True),
        StructField("day", IntegerType(), True),
    ])
    return spark.createDataFrame(results, schema)
